In [ ]:
import pandas as pd

def load_pairwise_csv(path, tag):
    df = pd.read_csv(path)

    # Normalize column names
    df = df.rename(columns={
        "y": "outcome",
        "Y": "outcome",
        "mediator": "mediator",
        "Mediator": "mediator"
    })

    required = [
        "outcome", "mediator",
        "acme", "acme_ci_low", "acme_ci_high", "acme_p",
        "ade", "ade_ci_low", "ade_ci_high", "ade_p"
    ]

    for col in required:
        if col not in df.columns:
            raise KeyError(f"Column missing: {col} in {path}")

    df = df.rename(columns={
        "acme": f"acme_{tag}",
        "acme_ci_low": f"acme_ci_low_{tag}",
        "acme_ci_high": f"acme_ci_high_{tag}",
        "acme_p": f"acme_p_{tag}",

        "ade": f"ade_{tag}",
        "ade_ci_low": f"ade_ci_low_{tag}",
        "ade_ci_high": f"ade_ci_high_{tag}",
        "ade_p": f"ade_p_{tag}",
    })

    return df


def merge_three(df01, df02, df12):
    return (
        df01.merge(df02, on=["outcome", "mediator"], how="outer")
            .merge(df12, on=["outcome", "mediator"], how="outer")
    )


def is_significant(row):
    """Return True if *any* ACME or ADE p-value < 0.05."""
    pcols = [
        "acme_p_01", "acme_p_02", "acme_p_12",
    ]
    return any((row[c] < 0.05) for c in pcols)


def latex_block(acme, lo, hi, p, ade, lo2, hi2, p2):
    sig1 = "*" if p < 0.05 else "†" if p < 0.10 else ""
    sig2 = "*" if p2 < 0.05 else "†" if p2 < 0.10 else ""

    return (
        f"ACME: {acme:.4f}{sig1} (p={p:.4f}) \\\\ "
        f"CI: [{lo:.4f}, {hi:.4f}] \\\\ "
        f"ADE: {ade:.4f}{sig2} (p={p2:.4f}) \\\\ "
        f"CI: [{lo2:.4f}, {hi2:.4f}]"
    )


def make_latex_rows(df):
    rows = []
    for _, r in df.iterrows():

        cell01 = latex_block(
            r["acme_01"], r["acme_ci_low_01"], r["acme_ci_high_01"], r["acme_p_01"],
            r["ade_01"],  r["ade_ci_low_01"],  r["ade_ci_high_01"],  r["ade_p_01"]
        )

        cell02 = latex_block(
            r["acme_02"], r["acme_ci_low_02"], r["acme_ci_high_02"], r["acme_p_02"],
            r["ade_02"],  r["ade_ci_low_02"],  r["ade_ci_high_02"],  r["ade_p_02"]
        )

        cell12 = latex_block(
            r["acme_12"], r["acme_ci_low_12"], r["acme_ci_high_12"], r["acme_p_12"],
            r["ade_12"],  r["ade_ci_low_12"],  r["ade_ci_high_12"],  r["ade_p_12"]
        )

        row = (
            f"{r['outcome']} & {r['mediator']} & "
            f"{cell01} & {cell02} & {cell12} \\\\ \\midrule"
        )

        rows.append(row)

    return rows


# -------------------------
# Run
# -------------------------

df01 = load_pairwise_csv("../results/mediation/pairwise_mediation_0_vs_1.csv", "01")
df02 = load_pairwise_csv("../results/mediation/pairwise_mediation_0_vs_2.csv", "02")
df12 = load_pairwise_csv("../results/mediation/pairwise_mediation_1_vs_2.csv", "12")

merged = merge_three(df01, df02, df12)

# Filter to only significant mediators
sig = merged[merged.apply(is_significant, axis=1)]

latex_rows = make_latex_rows(sig)

for row in latex_rows:
    print(row)
    print()


In [ ]:
from pathlib import Path
import pandas as pd

# ---- 1) Set paths (edit these) ----
csv_01 = Path("../results/mediation/pairwise_mediation_0_vs_1.csv")
csv_02 = Path("../results/mediation/pairwise_mediation_0_vs_2.csv")
csv_12 = Path("../results/mediation/pairwise_mediation_1_vs_2.csv")

out_csv = Path("../results/mediation/pairwise_mediation_raw_sig_acme.csv")
p_cut = 0.05  # keep pairs where any contrast has acme_p < p_cut

# ---- 2) Required columns ----
required = [
    "y", "mediator",
    "acme", "acme_p", "acme_ci_low", "acme_ci_high",
    "ade", "ade_p", "ade_ci_low", "ade_ci_high",
]

optional = [
    "total", "total_p", "total_ci_low", "total_ci_high",
    "prop_med",
    # if you added these in R, they will be kept automatically if present:
    "a_hat", "a_p", "b_hat", "b_p", "cprime_hat", "cprime_p", "c_hat", "c_p",
    "m_mean_g1", "m_mean_g2", "y_mean_g1", "y_mean_g2",
]

def load_one(path: Path, contrast: str) -> pd.DataFrame:
    if not path.exists():
        raise ValueError(f"Missing file: {path}")
    df = pd.read_csv(path)
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")
    df = df.copy()
    df["contrast"] = contrast
    return df

# ---- 3) Load + stack ----
df01 = load_one(csv_01, "0_vs_1")
df02 = load_one(csv_02, "0_vs_2")
df12 = load_one(csv_12, "1_vs_2")
df = pd.concat([df01, df02, df12], ignore_index=True)

# Keep only needed columns (required + whatever optional exists)
keep_cols = ["y", "mediator", "contrast"] + [c for c in required if c not in ("y", "mediator")]
keep_cols += [c for c in optional if c in df.columns]
df = df[keep_cols]

# Fail fast: duplicates per (y, mediator, contrast)
if df.duplicated(["y", "mediator", "contrast"]).any():
    dups = df[df.duplicated(["y", "mediator", "contrast"], keep=False)].sort_values(["y","mediator","contrast"])
    raise ValueError(f"Duplicate (y, mediator, contrast) rows found:\n{dups[['y','mediator','contrast']].head(20)}")

# ---- 4) Filter to pairs with any significant/marginal indirect effect ----
min_p = df.groupby(["y", "mediator"], dropna=False)["acme_p"].min().reset_index(name="min_acme_p")
keep_pairs = min_p[min_p["min_acme_p"] < p_cut][["y", "mediator"]]
df_sig = df.merge(keep_pairs, on=["y", "mediator"], how="inner").copy()

# ---- 5) Sort + save ----
df_sig = df_sig.sort_values(["y", "mediator", "contrast"]).reset_index(drop=True)
df_sig.to_csv(out_csv, index=False)

print("Saved:", out_csv, "| rows:", len(df_sig))
df_sig[["y","mediator","contrast","acme","acme_p","ade","ade_p"]].head(100)


In [ ]:
import numpy as np
import pandas as pd

# ---- fail-fast: need a_hat and b_hat to talk about "higher/lower mediator" and "higher/lower Y" ----
need = ["a_hat", "b_hat", "acme"]
missing = [c for c in need if c not in df_sig.columns]
if missing:
    raise ValueError(
        f"Missing columns needed for direction interpretation: {missing}\n"
        "Re-run the R script with a_hat and b_hat exported, or include them in the CSV."
    )

# Map contrast -> group labels (based on your coding in R script)
contrast_to_groups = {
    "0_vs_1": ("N", "A+P-"),
    "0_vs_2": ("N", "A+P+"),
    "1_vs_2": ("A+P-", "A+P+"),
}

def sgn(x):
    if pd.isna(x) or x == 0:
        return 0
    return 1 if x > 0 else -1

def dir_word(sign):
    return "higher" if sign > 0 else ("lower" if sign < 0 else "no_change")

def assoc_word(sign):
    return "higher" if sign > 0 else ("lower" if sign < 0 else "no_assoc")

def indirect_word(sign):
    return "increases" if sign > 0 else ("decreases" if sign < 0 else "no_indirect")

# Add group labels
df_sig = df_sig.copy()
df_sig["g1_label"] = df_sig["contrast"].map(lambda c: contrast_to_groups.get(c, ("?", "?"))[0])
df_sig["g2_label"] = df_sig["contrast"].map(lambda c: contrast_to_groups.get(c, ("?", "?"))[1])

# Direction pieces
df_sig["a_sign"] = df_sig["a_hat"].apply(sgn)          # g2 vs g1 effect on mediator
df_sig["b_sign"] = df_sig["b_hat"].apply(sgn)          # mediator association with outcome (adj.)
df_sig["ab_sign"] = (df_sig["a_hat"] * df_sig["b_hat"]).apply(sgn)
df_sig["acme_sign"] = df_sig["acme"].apply(sgn)

# Optional consistency check: sign(a*b) should match sign(ACME) most of the time
# (It can differ in edge cases due to simulation noise, nonlinear models, etc.)
mismatch = df_sig[(df_sig["ab_sign"] != 0) & (df_sig["acme_sign"] != 0) & (df_sig["ab_sign"] != df_sig["acme_sign"])]
print("Sign mismatches (a*b vs ACME):", len(mismatch))

# Plain-language direction strings
df_sig["gene_to_mediator_direction"] = df_sig.apply(
    lambda r: f"{r['g2_label']} has {dir_word(r['a_sign'])} {r['mediator']} than {r['g1_label']} (a_hat={r['a_hat']:.4f})",
    axis=1
)

df_sig["mediator_to_outcome_direction"] = df_sig.apply(
    lambda r: f"Higher {r['mediator']} is associated with {assoc_word(r['b_sign'])} {r['y']} (b_hat={r['b_hat']:.4f})",
    axis=1
)

df_sig["indirect_effect_direction"] = df_sig.apply(
    lambda r: f"Indirect effect via {r['mediator']} {indirect_word(r['acme_sign'])} {r['y']} for {r['g2_label']} vs {r['g1_label']} (ACME={r['acme']:.4f})",
    axis=1
)

# View the key columns
cols_show = [
    "y","mediator","contrast","g1_label","g2_label",
    "acme","acme_p","ade","ade_p",
    "a_hat","a_p","b_hat","b_p",
    "gene_to_mediator_direction",
    "mediator_to_outcome_direction",
    "indirect_effect_direction",
]
cols_show = [c for c in cols_show if c in df_sig.columns]
df_sig[cols_show].head(50)

# save updated CSV with direction columns included
df_sig.to_csv(out_csv, index=False)


In [ ]:
from pathlib import Path
import pandas as pd

# Inputs
csv_path = Path("../results/mediation/pairwise_mediation_raw_sig_acme.csv")  # change if needed
alpha = 0.05
alpha_marg = 0.10

# Load
df = pd.read_csv(csv_path)

# Fail fast: required cols
required = ["y", "mediator", "contrast", "acme_p"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Significance flags
df = df.copy()
df["sig_05"] = df["acme_p"] < alpha
df["sig_10"] = df["acme_p"] < alpha_marg

# 1) Unique mediator pairs significant anywhere (across contrasts)
pair_any_05 = df.groupby(["y", "mediator"], dropna=False)["sig_05"].any()
pair_any_10 = df.groupby(["y", "mediator"], dropna=False)["sig_10"].any()

print("Unique (y, mediator) pairs with ACME p < 0.05 anywhere:", int(pair_any_05.sum()))
print("Unique (y, mediator) pairs with ACME p < 0.10 anywhere:", int(pair_any_10.sum()))

# 2) Counts by contrast (unique pairs)
by_contrast_05 = (
    df[df["sig_05"]]
    .drop_duplicates(["y", "mediator", "contrast"])
    .groupby("contrast")
    .size()
    .sort_values(ascending=False)
)

by_contrast_10 = (
    df[df["sig_10"]]
    .drop_duplicates(["y", "mediator", "contrast"])
    .groupby("contrast")
    .size()
    .sort_values(ascending=False)
)

print("\nUnique (y, mediator) pairs significant per contrast (p < 0.05):")
print(by_contrast_05.to_string())

print("\nUnique (y, mediator) pairs marginal/significant per contrast (p < 0.10):")
print(by_contrast_10.to_string())

# 3) If you also want "how many distinct mediators" (ignoring outcome y)
med_any_05 = df.groupby(["mediator"], dropna=False)["sig_05"].any()
med_any_10 = df.groupby(["mediator"], dropna=False)["sig_10"].any()

print("\nDistinct mediators with ACME p < 0.05 anywhere:", int(med_any_05.sum()))
print("Distinct mediators with ACME p < 0.10 anywhere:", int(med_any_10.sum()))
